# Stage 1 baselines

This notebook implements the revised Stage 1 thesis design with three experimental conditions:

1. **Weak-only training**
2. **Gold-only training**
3. **Hybrid weak + gold training**

Gold labels are always used for evaluation on a held-out **Gold Test Set**.

Interpretation:
- `RMSE / MAE / R²` against `renegotiation_prob` are **weak-label distillation diagnostics**
- real Stage 1 validity is assessed on held-out **gold labels**

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

project_root: /Users/Thomas/Desktop/Master Thesis
src_path: /Users/Thomas/Desktop/Master Thesis/src


In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from master_thesis.config import MODELS_STAGE1, TABLES, SEED
from master_thesis.data_utils import (
    load_processed,
    merge_gold_labels,
    make_gold_contract_split,
)
from master_thesis.baselines import (
    fit_mean_predictor,
    fit_elastic_net,
    fit_elastic_net_weighted,
    fit_xgboost_regressor,
    fit_xgboost_regressor_weighted,
    extract_xgb_feature_importance,
)
from master_thesis.mlp import (
    set_seed,
    build_tabular_tensors,
    make_dataloaders,
    make_weighted_dataloaders,
    build_mlp,
    train_mlp,
    predict_mlp,
)
from master_thesis.metrics import (
    evaluate_predictions,
    evaluate_on_gold_binary,
)
from master_thesis.plotting import (
    plot_baseline_metric,
    plot_predicted_vs_true,
    plot_condition_metric,
    plot_feature_importance,
    save_fig,
)

## Load inputs

Expected processed files:
- `contract_snorkel_labeled.csv`
- `gold_labels.csv`

Update the filenames below if your project uses a different gold-label file name.

In [5]:
df_weak = load_processed("contract_with_features_labeled.csv", low_memory=False)
#df_gold = load_processed("gold_labels.csv", low_memory=False)

print("Weak-labeled shape:", df_weak.shape)
#print("Gold label shape:", df_gold.shape)

display(df_weak.head())
#display(df_gold.head())

Weak-labeled shape: (9201, 224)


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,global_esg_yes_votes,global_esg_no_votes,global_news_yes_votes,global_news_no_votes,global_market_yes_votes,global_market_no_votes,global_supplier_macro_yes_votes,global_supplier_macro_no_votes,logistics_specific_yes_votes,logistics_specific_no_votes
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,0,0,0,0
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,0,0,0,0
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,0,0,0,0
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,0,0,0,0
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756,0,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,0,0,0,0,0,0,1,0,0,0


## Merge gold labels and create Gold Train / Gold Test split

In [ ]:
df_stage1 = merge_gold_labels(df_weak, df_gold)

gold_train_contract_ids, gold_test_contract_ids = make_gold_contract_split(
    df_stage1=df_stage1,
    seed=SEED,
    test_size=0.30,
)

print("Merged Stage 1 shape:", df_stage1.shape)
print("Gold train contracts:", len(gold_train_contract_ids))
print("Gold test contracts :", len(gold_test_contract_ids))
display(df_stage1.head())

## Stage 1 modeling setup

In [ ]:
df_baseline = df_stage1.copy()

weak_target_col = "renegotiation_prob"
gold_target_col = "gold_y"
group_col = "contract_id"

drop_cols = [
    "Unnamed: 0",
    "contract_id",
    "contract_number",
    "contract_name",
    "supplier_id",
    "supplier_number",
    "supplier_display_name",
    "moodys_bvd_id",
    "Company name Latin alphabet",
    "target_renegotiate",
    "start_date",
    "expiration_date",
    "gold_y",
]

feature_cols = [
    c for c in df_baseline.columns
    if c not in drop_cols + [weak_target_col]
    and not c.endswith("_yes_votes")
    and not c.endswith("_no_votes")
    and c != "lf_abstain_votes"
]

X_all = df_baseline[feature_cols].copy()

numeric_cols = X_all.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
categorical_cols = X_all.select_dtypes(include=["object", "string", "str"]).columns.tolist()

print("Feature count:", X_all.shape[1])
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

## Condition A — Weak-only training

Train on weak labels only, excluding the held-out Gold Test Set from training.
Evaluate:
- weak-label distillation fit on the Gold Test rows using `renegotiation_prob`
- real gold-label validity on the Gold Test rows using `gold_y`

In [ ]:
df_weak_train = df_baseline[~df_baseline["contract_id"].isin(gold_test_contract_ids)].copy()
df_gold_test_eval = df_baseline[df_baseline["contract_id"].isin(gold_test_contract_ids)].copy()

X_weak_train = df_weak_train[feature_cols].copy()
y_weak_train = df_weak_train[weak_target_col].astype(float).copy()

X_gold_test = df_gold_test_eval[feature_cols].copy()
y_gold_test = df_gold_test_eval[gold_target_col].astype(int).copy()
y_weak_test = df_gold_test_eval[weak_target_col].astype(float).copy()

weak_distillation_results_list = []
weak_gold_eval_results_list = []

df_weak_test_predictions = pd.DataFrame({
    "contract_id": df_gold_test_eval["contract_id"].values,
    "y_true_weak": y_weak_test.values,
    "y_true_gold": y_gold_test.values,
})

### Mean Predictor

In [ ]:
y_pred_mean, df_weak_results_mean = fit_mean_predictor(y_train=y_weak_train, y_val=y_weak_test)
df_weak_results_mean["condition"] = "weak_only"
weak_distillation_results_list.append(df_weak_results_mean)

df_gold_eval_mean = evaluate_on_gold_binary(y_gold_test, y_pred_mean, model_name="Mean Predictor")
df_gold_eval_mean["condition"] = "weak_only"
weak_gold_eval_results_list.append(df_gold_eval_mean)

df_weak_test_predictions["mean_predictor"] = y_pred_mean

display(df_weak_results_mean)
display(df_gold_eval_mean)

### Elastic Net

In [ ]:
enet_model_weak, y_pred_enet, df_weak_results_enet = fit_elastic_net(
    X_train=X_weak_train,
    X_val=X_gold_test,
    y_train=y_weak_train,
    y_val=y_weak_test,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
)

df_weak_results_enet["condition"] = "weak_only"
weak_distillation_results_list.append(df_weak_results_enet)

df_gold_eval_enet = evaluate_on_gold_binary(y_gold_test, y_pred_enet, model_name="Elastic Net")
df_gold_eval_enet["condition"] = "weak_only"
weak_gold_eval_results_list.append(df_gold_eval_enet)

df_weak_test_predictions["elastic_net"] = y_pred_enet

display(df_weak_results_enet)
display(df_gold_eval_enet)

### XGBoost

In [ ]:
xgb_model_weak, y_pred_xgb_weak, df_weak_results_xgb = fit_xgboost_regressor(
    X_train=X_weak_train,
    X_val=X_gold_test,
    y_train=y_weak_train,
    y_val=y_weak_test,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
)

df_weak_results_xgb["condition"] = "weak_only"
weak_distillation_results_list.append(df_weak_results_xgb)

df_gold_eval_xgb = evaluate_on_gold_binary(y_gold_test, y_pred_xgb_weak, model_name="XGBoost")
df_gold_eval_xgb["condition"] = "weak_only"
weak_gold_eval_results_list.append(df_gold_eval_xgb)

df_weak_test_predictions["xgboost"] = y_pred_xgb_weak

display(df_weak_results_xgb)
display(df_gold_eval_xgb)

### MLP

In [ ]:
set_seed(SEED)

X_weak_train_tensor, X_gold_test_tensor, y_weak_train_tensor, y_weak_test_tensor = build_tabular_tensors(
    preprocessor=preprocessor,
    X_train=X_weak_train,
    X_val=X_gold_test,
    y_train=y_weak_train,
    y_val=y_weak_test,
)

weak_train_loader, weak_test_loader = make_dataloaders(
    X_train_tensor=X_weak_train_tensor,
    X_val_tensor=X_gold_test_tensor,
    y_train_tensor=y_weak_train_tensor,
    y_val_tensor=y_weak_test_tensor,
    train_batch_size=128,
    val_batch_size=256,
)

mlp_weak, criterion_weak, optimizer_weak, device = build_mlp(
    input_dim=X_weak_train_tensor.shape[1],
    lr=1e-3,
    weight_decay=1e-5,
)

mlp_weak, df_mlp_weak_history = train_mlp(
    model=mlp_weak,
    train_loader=weak_train_loader,
    val_loader=weak_test_loader,
    criterion=criterion_weak,
    optimizer=optimizer_weak,
    device=device,
    n_epochs=50,
    patience=5,
    verbose=True,
)

y_pred_mlp_weak = predict_mlp(mlp_weak, weak_test_loader, device)

df_weak_results_mlp = evaluate_predictions(
    y_true=y_weak_test,
    y_pred=y_pred_mlp_weak,
    model_name="MLP",
)
df_weak_results_mlp["condition"] = "weak_only"
weak_distillation_results_list.append(df_weak_results_mlp)

df_gold_eval_mlp = evaluate_on_gold_binary(y_gold_test, y_pred_mlp_weak, model_name="MLP")
df_gold_eval_mlp["condition"] = "weak_only"
weak_gold_eval_results_list.append(df_gold_eval_mlp)

df_weak_test_predictions["mlp"] = y_pred_mlp_weak

display(df_weak_results_mlp)
display(df_gold_eval_mlp)

In [ ]:
MODELS_STAGE1.mkdir(parents=True, exist_ok=True)

torch.save(mlp_weak.state_dict(), MODELS_STAGE1 / "stage1_mlp_weak_only.pt")
joblib.dump(preprocessor, MODELS_STAGE1 / "stage1_mlp_weak_only_preprocessor.joblib")
df_mlp_weak_history.to_csv(TABLES / "stage1_mlp_weak_only_history.csv", index=False)

## Condition B — Gold-only training

Train only on gold-labeled contracts from the Gold Train split.
Evaluate on the held-out Gold Test Set.

In [ ]:
df_gold_train = df_baseline[df_baseline["contract_id"].isin(gold_train_contract_ids)].copy()
df_gold_test = df_baseline[df_baseline["contract_id"].isin(gold_test_contract_ids)].copy()

X_gold_train = df_gold_train[feature_cols].copy()
y_gold_train = df_gold_train[gold_target_col].astype(int).copy()

X_gold_test = df_gold_test[feature_cols].copy()
y_gold_test = df_gold_test[gold_target_col].astype(int).copy()

gold_only_results_list = []

### Mean Predictor

In [ ]:
y_pred_mean_gold, _ = fit_mean_predictor(y_train=y_gold_train, y_val=y_gold_test)

df_gold_only_eval_mean = evaluate_on_gold_binary(y_gold_test, y_pred_mean_gold, model_name="Mean Predictor")
df_gold_only_eval_mean["condition"] = "gold_only"
gold_only_results_list.append(df_gold_only_eval_mean)

display(df_gold_only_eval_mean)

### Elastic Net

In [ ]:
enet_model_gold, y_pred_enet_gold, _ = fit_elastic_net(
    X_train=X_gold_train,
    X_val=X_gold_test,
    y_train=y_gold_train,
    y_val=y_gold_test,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
)

df_gold_only_eval_enet = evaluate_on_gold_binary(y_gold_test, y_pred_enet_gold, model_name="Elastic Net")
df_gold_only_eval_enet["condition"] = "gold_only"
gold_only_results_list.append(df_gold_only_eval_enet)

display(df_gold_only_eval_enet)

### XGBoost

In [ ]:
xgb_model_gold, y_pred_xgb_gold, _ = fit_xgboost_regressor(
    X_train=X_gold_train,
    X_val=X_gold_test,
    y_train=y_gold_train,
    y_val=y_gold_test,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
)

df_gold_only_eval_xgb = evaluate_on_gold_binary(y_gold_test, y_pred_xgb_gold, model_name="XGBoost")
df_gold_only_eval_xgb["condition"] = "gold_only"
gold_only_results_list.append(df_gold_only_eval_xgb)

display(df_gold_only_eval_xgb)

### MLP

In [ ]:
set_seed(SEED)

X_gold_train_tensor, X_gold_test_tensor, y_gold_train_tensor, y_gold_test_tensor = build_tabular_tensors(
    preprocessor=preprocessor,
    X_train=X_gold_train,
    X_val=X_gold_test,
    y_train=y_gold_train,
    y_val=y_gold_test,
)

gold_train_loader, gold_test_loader = make_dataloaders(
    X_train_tensor=X_gold_train_tensor,
    X_val_tensor=X_gold_test_tensor,
    y_train_tensor=y_gold_train_tensor,
    y_val_tensor=y_gold_test_tensor,
    train_batch_size=64,
    val_batch_size=256,
)

mlp_gold, criterion_gold, optimizer_gold, device = build_mlp(
    input_dim=X_gold_train_tensor.shape[1],
    lr=1e-3,
    weight_decay=1e-5,
)

mlp_gold, df_mlp_gold_history = train_mlp(
    model=mlp_gold,
    train_loader=gold_train_loader,
    val_loader=gold_test_loader,
    criterion=criterion_gold,
    optimizer=optimizer_gold,
    device=device,
    n_epochs=50,
    patience=5,
    verbose=True,
)

y_pred_mlp_gold = predict_mlp(mlp_gold, gold_test_loader, device)

df_gold_only_eval_mlp = evaluate_on_gold_binary(y_gold_test, y_pred_mlp_gold, model_name="MLP")
df_gold_only_eval_mlp["condition"] = "gold_only"
gold_only_results_list.append(df_gold_only_eval_mlp)

display(df_gold_only_eval_mlp)

In [ ]:
torch.save(mlp_gold.state_dict(), MODELS_STAGE1 / "stage1_mlp_gold_only.pt")
df_mlp_gold_history.to_csv(TABLES / "stage1_mlp_gold_only_history.csv", index=False)

## Condition C — Hybrid weak + gold training

Implementation:
- weak labels for all non-test rows
- gold-labeled train contracts overwrite weak targets with gold targets
- gold train rows receive higher weight

In [ ]:
df_hybrid_train = df_baseline[~df_baseline["contract_id"].isin(gold_test_contract_ids)].copy()
df_hybrid_test = df_baseline[df_baseline["contract_id"].isin(gold_test_contract_ids)].copy()

df_hybrid_train["train_target"] = df_hybrid_train[weak_target_col].astype(float)
df_hybrid_train["sample_weight"] = 1.0

gold_train_mask = df_hybrid_train["contract_id"].isin(gold_train_contract_ids) & df_hybrid_train["gold_y"].notna()
df_hybrid_train.loc[gold_train_mask, "train_target"] = df_hybrid_train.loc[gold_train_mask, "gold_y"].astype(float)
df_hybrid_train.loc[gold_train_mask, "sample_weight"] = 3.0

X_hybrid_train = df_hybrid_train[feature_cols].copy()
y_hybrid_train = df_hybrid_train["train_target"].astype(float).copy()
w_hybrid_train = df_hybrid_train["sample_weight"].astype(float).copy()

X_hybrid_test = df_hybrid_test[feature_cols].copy()
y_hybrid_test_gold = df_hybrid_test[gold_target_col].astype(int).copy()

hybrid_results_list = []

### Mean Predictor

In [ ]:
weighted_mean = np.average(y_hybrid_train, weights=w_hybrid_train)
y_pred_mean_hybrid = np.full(shape=len(y_hybrid_test_gold), fill_value=weighted_mean)

df_hybrid_eval_mean = evaluate_on_gold_binary(y_hybrid_test_gold, y_pred_mean_hybrid, model_name="Mean Predictor")
df_hybrid_eval_mean["condition"] = "hybrid"
hybrid_results_list.append(df_hybrid_eval_mean)

display(df_hybrid_eval_mean)

### Elastic Net

In [ ]:
enet_model_hybrid, y_pred_enet_hybrid, _ = fit_elastic_net_weighted(
    X_train=X_hybrid_train,
    X_val=X_hybrid_test,
    y_train=y_hybrid_train,
    y_val=y_hybrid_test_gold,
    sample_weight=w_hybrid_train,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
)

df_hybrid_eval_enet = evaluate_on_gold_binary(y_hybrid_test_gold, y_pred_enet_hybrid, model_name="Elastic Net")
df_hybrid_eval_enet["condition"] = "hybrid"
hybrid_results_list.append(df_hybrid_eval_enet)

display(df_hybrid_eval_enet)

### XGBoost

In [ ]:
xgb_model_hybrid, y_pred_xgb_hybrid, _ = fit_xgboost_regressor_weighted(
    X_train=X_hybrid_train,
    X_val=X_hybrid_test,
    y_train=y_hybrid_train,
    y_val=y_hybrid_test_gold,
    sample_weight=w_hybrid_train,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
)

df_hybrid_eval_xgb = evaluate_on_gold_binary(y_hybrid_test_gold, y_pred_xgb_hybrid, model_name="XGBoost")
df_hybrid_eval_xgb["condition"] = "hybrid"
hybrid_results_list.append(df_hybrid_eval_xgb)

display(df_hybrid_eval_xgb)

### MLP

In [ ]:
set_seed(SEED)

X_hybrid_train_tensor, X_hybrid_test_tensor, y_hybrid_train_tensor, y_hybrid_test_tensor = build_tabular_tensors(
    preprocessor=preprocessor,
    X_train=X_hybrid_train,
    X_val=X_hybrid_test,
    y_train=y_hybrid_train,
    y_val=y_hybrid_test_gold,
)

hybrid_train_loader, hybrid_test_loader = make_weighted_dataloaders(
    X_train_tensor=X_hybrid_train_tensor,
    X_val_tensor=X_hybrid_test_tensor,
    y_train_tensor=y_hybrid_train_tensor,
    y_val_tensor=y_hybrid_test_tensor,
    sample_weight=w_hybrid_train,
    train_batch_size=128,
    val_batch_size=256,
)

mlp_hybrid, criterion_hybrid, optimizer_hybrid, device = build_mlp(
    input_dim=X_hybrid_train_tensor.shape[1],
    lr=1e-3,
    weight_decay=1e-5,
)

mlp_hybrid, df_mlp_hybrid_history = train_mlp(
    model=mlp_hybrid,
    train_loader=hybrid_train_loader,
    val_loader=hybrid_test_loader,
    criterion=criterion_hybrid,
    optimizer=optimizer_hybrid,
    device=device,
    n_epochs=50,
    patience=5,
    verbose=True,
)

y_pred_mlp_hybrid = predict_mlp(mlp_hybrid, hybrid_test_loader, device)

df_hybrid_eval_mlp = evaluate_on_gold_binary(y_hybrid_test_gold, y_pred_mlp_hybrid, model_name="MLP")
df_hybrid_eval_mlp["condition"] = "hybrid"
hybrid_results_list.append(df_hybrid_eval_mlp)

display(df_hybrid_eval_mlp)

In [ ]:
torch.save(mlp_hybrid.state_dict(), MODELS_STAGE1 / "stage1_mlp_hybrid.pt")
df_mlp_hybrid_history.to_csv(TABLES / "stage1_mlp_hybrid_history.csv", index=False)

## Consolidated Stage 1 outputs

In [ ]:
df_weak_distillation_results = pd.concat(weak_distillation_results_list, ignore_index=True)
df_gold_evaluation_results = pd.concat(
    weak_gold_eval_results_list + gold_only_results_list + hybrid_results_list,
    ignore_index=True,
)

df_weak_distillation_results = df_weak_distillation_results.sort_values("mae", ascending=True).reset_index(drop=True)
df_gold_evaluation_results = df_gold_evaluation_results.sort_values(
    ["condition", "gold_logloss"], ascending=[True, True]
).reset_index(drop=True)

display(df_weak_distillation_results)
display(df_gold_evaluation_results)

## Save outputs

In [ ]:
df_weak_distillation_results.to_csv(TABLES / "stage1_weak_distillation_results.csv", index=False)
df_gold_evaluation_results.to_csv(TABLES / "stage1_gold_evaluation_results.csv", index=False)
df_weak_test_predictions.to_csv(TABLES / "stage1_weak_only_test_predictions.csv", index=False)

df_xgb_importance_weak = extract_xgb_feature_importance(xgb_model_weak)
df_xgb_importance_weak.to_csv(TABLES / "stage1_xgb_weak_only_feature_importance.csv", index=False)

display(df_xgb_importance_weak.head(20))

print("Saved Stage 1 outputs.")

## Plots

In [ ]:
fig, ax = plot_baseline_metric(
    df_weak_distillation_results,
    metric="mae",
    title="Weak-only Distillation MAE",
)
save_fig(fig, "stage1_weak_only_distillation_mae.png")
plt.show()

fig, ax = plot_baseline_metric(
    df_weak_distillation_results,
    metric="rmse",
    title="Weak-only Distillation RMSE",
)
save_fig(fig, "stage1_weak_only_distillation_rmse.png")
plt.show()

fig, ax = plot_condition_metric(
    df_gold_evaluation_results,
    metric="gold_auroc",
    title="Stage 1 Gold AUROC by Condition",
)
save_fig(fig, "stage1_gold_auroc_by_condition.png")
plt.show()

fig, ax = plot_condition_metric(
    df_gold_evaluation_results,
    metric="gold_logloss",
    title="Stage 1 Gold Log-Loss by Condition",
)
save_fig(fig, "stage1_gold_logloss_by_condition.png")
plt.show()

fig, ax = plot_predicted_vs_true(
    y_weak_test,
    y_pred_xgb_weak,
    model_name="XGBoost Weak-only",
)
save_fig(fig, "stage1_xgb_weak_only_pred_vs_true.png")
plt.show()

fig, ax = plot_feature_importance(
    df_xgb_importance_weak,
    top_n=20,
    title="XGBoost Weak-only Feature Importance",
)
save_fig(fig, "stage1_xgb_weak_only_feature_importance.png")
plt.show()

## Short interpretation

Use this section to summarize:
- which model best learns the weak-label signal
- which model best generalizes to held-out gold labels
- whether hybrid training improves over weak-only
- whether gold-only is unstable or competitive
- which MLP checkpoint should be used for Stage 2 fine-tuning and MAML comparison